# Layer 2 Backbone Exploration

This notebook runs the second-layer backbone comparison for the selected hard settings.

Fixed design:
- Architecture: `Unet`
- Encoder weights: `imagenet`
- Settings: `Single-TB`, `Shift-TA_TC-to-TB_10pct`, `Shift-TA_TB-to-TC_10pct`
- Backbones: `resnet34`, `efficientnet-b0`, `resnet50`


## 1. Setup

In [1]:
from pathlib import Path
import subprocess
import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path('/users/7/yu001011/csci5527/CSCI5527-final')
SCRIPT_DIR = PROJECT_ROOT / 'baseline_models_scripts'
RESULTS_DIR = SCRIPT_DIR / 'runs'
VENV_PY = Path('/users/7/yu001011/csci5527/.venv/bin/python')

PROJECT_ROOT, SCRIPT_DIR, RESULTS_DIR, VENV_PY

(PosixPath('/users/7/yu001011/csci5527/CSCI5527-final'),
 PosixPath('/users/7/yu001011/csci5527/CSCI5527-final/baseline_models_scripts'),
 PosixPath('/users/7/yu001011/csci5527/CSCI5527-final/baseline_models_scripts/runs'),
 PosixPath('/users/7/yu001011/csci5527/.venv/bin/python'))

## 2. Experiment Design

In [2]:
ARCH = 'Unet'
ENCODER_WEIGHTS = 'imagenet'
EPOCHS = 100
BATCH_SIZE = 16
IMG_SIZE = 512
LR = 1e-4

SETTINGS = [
    'Single-TB',
    'Shift-TA_TC-to-TB_10pct',
    'Shift-TA_TB-to-TC_10pct',
]

BACKBONES = [
#     'resnet34',
    'efficientnet-b0',
    'resnet50',
]

pd.DataFrame([
    {'Setting': setting, 'Backbone': backbone}
    for setting in SETTINGS
    for backbone in BACKBONES
])

,Setting,Backbone
0,Single-TB,efficientnet-b0
1,Single-TB,resnet50
2,Shift-TA_TC-to-TB_10pct,efficientnet-b0
3,Shift-TA_TC-to-TB_10pct,resnet50
4,Shift-TA_TB-to-TC_10pct,efficientnet-b0
5,Shift-TA_TB-to-TC_10pct,resnet50


## 3. Helper Functions

In [3]:
def result_filename(backbone: str) -> str:
    return f'layer2_{backbone}.csv'

def build_command(setting: str, backbone: str):
    return [
        str(VENV_PY),
        'train_ttd.py',
        '--project-root', str(PROJECT_ROOT),
        '--experiment', setting,
        '--arch', ARCH,
        '--encoder', backbone,
        '--encoder-weights', ENCODER_WEIGHTS,
        '--epochs', str(EPOCHS),
        '--batch-size', str(BATCH_SIZE),
        '--img-size', str(IMG_SIZE),
        '--lr', str(LR),
        '--results-name', result_filename(backbone),
    ]

def run_one(setting: str, backbone: str):
    cmd = build_command(setting, backbone)
    print('Running:', ' '.join(cmd))
    return subprocess.run(cmd, cwd=SCRIPT_DIR, check=True)

def run_all(settings, backbones):
    for backbone in backbones:
        for setting in settings:
            run_one(setting, backbone)


## 4. Preview Commands

Use this first to confirm the exact commands before running anything.

In [4]:
command_preview = pd.DataFrame([
    {
        'Setting': setting,
        'Backbone': backbone,
        'Command': ' '.join(build_command(setting, backbone)),
    }
    for setting in SETTINGS
    for backbone in BACKBONES
])

display(command_preview)

,Setting,Backbone,Command
0,Single-TB,efficientnet-b0,/users/7/yu001011/csci5527/.venv/bin/python tr...
1,Single-TB,resnet50,/users/7/yu001011/csci5527/.venv/bin/python tr...
2,Shift-TA_TC-to-TB_10pct,efficientnet-b0,/users/7/yu001011/csci5527/.venv/bin/python tr...
3,Shift-TA_TC-to-TB_10pct,resnet50,/users/7/yu001011/csci5527/.venv/bin/python tr...
4,Shift-TA_TB-to-TC_10pct,efficientnet-b0,/users/7/yu001011/csci5527/.venv/bin/python tr...
5,Shift-TA_TB-to-TC_10pct,resnet50,/users/7/yu001011/csci5527/.venv/bin/python tr...


## 5. Run a Single Experiment

Start here if you want to test one configuration first.

In [5]:
TEST_SETTING = 'Single-TB'
TEST_BACKBONE = 'resnet34'

# Uncomment to run:
# run_one(TEST_SETTING, TEST_BACKBONE)

In [6]:
! /users/7/yu001011/csci5527/.venv/bin/python -m pip install pandas

## 6. Run the Full Layer 2 Grid

This runs all 9 experiments.

Warning: this may take a long time depending on GPU availability.

In [7]:
# Uncomment to run all experiments:
run_all(SETTINGS, BACKBONES)

Running: /users/7/yu001011/csci5527/.venv/bin/python train_ttd.py --project-root /users/7/yu001011/csci5527/CSCI5527-final --experiment Single-TB --arch Unet --encoder efficientnet-b0 --encoder-weights imagenet --epochs 100 --batch-size 16 --img-size 512 --lr 0.0001 --results-name layer2_efficientnet-b0.csv
Python executable: /users/7/yu001011/csci5527/.venv/bin/python
Torch version: 2.11.0+cu126
CUDA available: True
CUDA device count: 1
GPU 0: NVIDIA H100
Using device: cuda:0

===== Running Single-TB | arch=Unet | encoder=efficientnet-b0 =====
Loading CSV metadata...
Sanitizing training and validation masks...


Sanitizing Masks: 100%|██████████| 48/48 [00:00<00:00, 4479.70it/s]


Sanitizing test masks...


Generating Dataloaders...

Pipeline Ready:
 - Training batches: 20
 - Validation batches: 6
 - Testing batches: 3
  → Unet-efficientnet-b0-imagenet_Single-TB:  41%|████      | 41/100 [04:20<06:19,  6.43s/it, IoU=0.3750, F1=0.5386]Epoch 41: T-Loss: 6.1325 | V-Loss: 5.8989 | IoU: 0.3750 | F1: 0.5386 [Saved Best Model]


  → Unet-efficientnet-b0-imagenet_Single-TB:  66%|██████▌   | 66/100 [06:58<03:40,  6.49s/it, IoU=0.3894, F1=0.5543]Epoch 66: T-Loss: 5.8653 | V-Loss: 5.8628 | IoU: 0.3894 | F1: 0.5543

Early Stopping Triggered at Epoch 66
  - Best Validation Loss: 5.8157
  - Patience Exhausted: 10/10 epochs without improvement
  - Stopping training now.
Charts saved to /users/7/yu001011/csci5527/CSCI5527-final/baseline_models_scripts/runs/figures/                     
Visualization saved to figures/Unet-efficientnet-b0-imagenet_Single-TB_visualization.png
Saved interim results to: /users/7/yu001011/csci5527/CSCI5527-final/baseline_models_scripts/runs/layer2_efficientnet-b0.csv

Final results:
                                Experiment  Val_Loss  ...  Test_Recall  Test_Prec
0  Unet-efficientnet-b0-imagenet_Single-TB  5.843002  ...     0.390885   0.486734

[1 rows x 11 columns]
Running: /users/7/yu001011/csci5527/.venv/bin/python train_ttd.py --project-root /users/7/yu001011/csci5527/CSCI5527-final --ex

...

Sanitizing Masks: 100%|██████████| 48/48 [00:00<00:00, 36785.42it/s]


Sanitizing test masks...
Generating Dataloaders...

Pipeline Ready:
 - Training batches: 45
 - Validation batches: 13
 - Testing batches: 3
  → Unet-resnet50-imagenet_Shift-TA_TC-to-TB_10pct:   0%|          | 0/100 [00:21<?, ?it/s, IoU=0.1466, F1=0.2444]Epoch 0: T-Loss: 9.8842 | V-Loss: 9.2023 | IoU: 0.1466 | F1: 0.2444 [Saved Best Model]
  → Unet-resnet50-imagenet_Shift-TA_TC-to-TB_10pct:   1%|          | 1/100 [00:38<36:35, 22.18s/it, IoU=0.2574, F1=0.3956]Epoch 1: T-Loss: 8.7073 | V-Loss: 8.4319 | IoU: 0.2574 | F1: 0.3956 [Saved Best Model]
  → Unet-resnet50-imagenet_Shift-TA_TC-to-TB_10pct:   2%|▏         | 2/100 [01:01<31:02, 19.01s/it, IoU=0.3247, F1=0.4705]Epoch 2: T-Loss: 8.3342 | V-Loss: 8.1197 | IoU: 0.3247 | F1: 0.4705 [Saved Best Model]
  → Unet-resnet50-imagenet_Shift-TA_TC-to-TB_10pct:   3%|▎         | 3/100 [01:20<33:42, 20.85s/it, IoU=0.3253, F1=0.4775]Epoch 3: T-Loss: 8.0501 | V-Loss: 7.9037 | IoU: 0.3253 | F1: 0.4775 [Saved Best Model]
  → Unet-resnet50-imagenet_Shift

  → Unet-resnet50-imagenet_Shift-TA_TC-to-TB_10pct:  41%|████      | 41/100 [14:45<18:32, 18.85s/it, IoU=0.5387, F1=0.6770]Epoch 41: T-Loss: 5.5878 | V-Loss: 5.6972 | IoU: 0.5387 | F1: 0.6770
  → Unet-resnet50-imagenet_Shift-TA_TC-to-TB_10pct:  42%|████▏     | 42/100 [15:02<17:17, 17.88s/it, IoU=0.5442, F1=0.6828]Epoch 42: T-Loss: 5.5582 | V-Loss: 5.6400 | IoU: 0.5442 | F1: 0.6828
  → Unet-resnet50-imagenet_Shift-TA_TC-to-TB_10pct:  43%|████▎     | 43/100 [15:27<16:47, 17.68s/it, IoU=0.5417, F1=0.6804]Epoch 43: T-Loss: 5.5210 | V-Loss: 5.6273 | IoU: 0.5417 | F1: 0.6804
  → Unet-resnet50-imagenet_Shift-TA_TC-to-TB_10pct:  44%|████▍     | 44/100 [15:44<18:22, 19.69s/it, IoU=0.5309, F1=0.6683]Epoch 44: T-Loss: 5.5575 | V-Loss: 5.7226 | IoU: 0.5309 | F1: 0.6683
  → Unet-resnet50-imagenet_Shift-TA_TC-to-TB_10pct:  45%|████▌     | 45/100 [16:02<17:28, 19.06s/it, IoU=0.5513, F1=0.6893]Epoch 45: T-Loss: 5.5434 | V-Loss: 5.7175 | IoU: 0.5513 | F1: 0.6893 [Saved Best Model]
  → Unet-resnet50-ima

Sanitizing Masks: 100%|██████████| 38/38 [00:00<00:00, 3827.47it/s]


Sanitizing test masks...
Generating Dataloaders...

Pipeline Ready:
 - Training batches: 50
 - Validation batches: 15
 - Testing batches: 3
  → Unet-resnet50-imagenet_Shift-TA_TB-to-TC_10pct:   0%|          | 0/100 [00:19<?, ?it/s, IoU=0.1928, F1=0.3187]Epoch 0: T-Loss: 9.3341 | V-Loss: 8.5501 | IoU: 0.1928 | F1: 0.3187 [Saved Best Model]
  → Unet-resnet50-imagenet_Shift-TA_TB-to-TC_10pct:   1%|          | 1/100 [00:38<32:38, 19.78s/it, IoU=0.2591, F1=0.4018]Epoch 1: T-Loss: 8.4067 | V-Loss: 8.0538 | IoU: 0.2591 | F1: 0.4018 [Saved Best Model]
  → Unet-resnet50-imagenet_Shift-TA_TB-to-TC_10pct:   2%|▏         | 2/100 [00:57<31:38, 19.37s/it, IoU=0.3514, F1=0.5022]Epoch 2: T-Loss: 7.9691 | V-Loss: 7.6264 | IoU: 0.3514 | F1: 0.5022 [Saved Best Model]
  → Unet-resnet50-imagenet_Shift-TA_TB-to-TC_10pct:   3%|▎         | 3/100 [01:16<31:01, 19.19s/it, IoU=0.3464, F1=0.5036]Epoch 3: T-Loss: 7.6279 | V-Loss: 7.4014 | IoU: 0.3464 | F1: 0.5036
  → Unet-resnet50-imagenet_Shift-TA_TB-to-TC_10pct:

## 7. Load Layer 2 Results

Run this after some or all experiments finish.

In [8]:
def load_layer2_results(backbones):
    frames = []
    for backbone in backbones:
        path = RESULTS_DIR / result_filename(backbone)
        if path.exists():
            df = pd.read_csv(path)
            df['Backbone'] = backbone
            frames.append(df)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)

layer2_df = load_layer2_results(BACKBONES)
layer2_df

,Experiment,Val_Loss,Val_IoU,Val_F1,Val_Recall,Val_Prec,Test_Loss,Test_IoU,Test_F1,Test_Recall,Test_Prec,Backbone
0,Unet-efficientnet-b0-imagenet_Shift-TA_TB-to-T...,5.420809,0.579151,0.725868,0.777058,0.697560,5.524915,0.377296,0.545845,0.586488,0.510535,efficientnet-b0
1,Unet-resnet50-imagenet_Shift-TA_TB-to-TC_10pct,5.629375,0.569046,0.714772,0.764933,0.692679,6.058224,0.264646,0.416388,0.433505,0.424765,resnet50


## 8. Clean Experiment Labels

In [9]:
if not layer2_df.empty:
    layer2_df = layer2_df.copy()
    layer2_df['Setting'] = layer2_df['Experiment'].str.replace(r'^Unet-[^-]+-imagenet_', '', regex=True)
    display(layer2_df[['Backbone', 'Setting', 'Test_IoU', 'Test_F1', 'Test_Recall', 'Test_Prec']])
else:
    display(Markdown('No Layer 2 result files found yet.'))

,Backbone,Setting,Test_IoU,Test_F1,Test_Recall,Test_Prec
0,efficientnet-b0,Unet-efficientnet-b0-imagenet_Shift-TA_TB-to-T...,0.377296,0.545845,0.586488,0.510535
1,resnet50,Shift-TA_TB-to-TC_10pct,0.264646,0.416388,0.433505,0.424765


## 9. Pivot Table for Comparison

In [10]:
if not layer2_df.empty:
    pivot_iou = layer2_df.pivot(index='Setting', columns='Backbone', values='Test_IoU')
    pivot_f1 = layer2_df.pivot(index='Setting', columns='Backbone', values='Test_F1')
    display(Markdown('### Test IoU'))
    display(pivot_iou)
    display(Markdown('### Test F1'))
    display(pivot_f1)
else:
    display(Markdown('No results available to pivot yet.'))

### Test IoU

Backbone,efficientnet-b0,resnet50
Setting,,
Shift-TA_TB-to-TC_10pct,NaN,0.264646
Unet-efficientnet-b0-imagenet_Shift-TA_TB-to-TC_10pct,0.377296,NaN


### Test F1

Backbone,efficientnet-b0,resnet50
Setting,,
Shift-TA_TB-to-TC_10pct,NaN,0.416388
Unet-efficientnet-b0-imagenet_Shift-TA_TB-to-TC_10pct,0.545845,NaN


## 10. Quick Discussion Prompts

Use these after results are loaded:

- Which backbone gives the best `Test_IoU` on `Single-TB`?
- Does `efficientnet-b0` improve the domain-shift settings compared with `resnet34`?
- Does `resnet50` help because of greater depth, or is `efficientnet-b0` still stronger?
- Are the gains mostly from recall, precision, or both?
